# 🚗 Car Price Prediction with Machine Learning

End-to-end supervised regression pipeline — from raw data to a saved production-ready model.

**Dataset:** 301 used car listings (India) | **Target:** Selling Price (Lakhs INR)

---

## 0. Setup — imports and paths

In [ ]:
import warnings, os
from pathlib import Path
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
RANDOM_STATE = 42

ROOT = Path(".")
DATA_PATH   = ROOT / "data" / "car_data.csv"
PLOTS_DIR   = ROOT / "outputs" / "plots"
MODELS_DIR  = ROOT / "models"
for p in [PLOTS_DIR, ROOT / "outputs" / "reports", MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print("Setup complete.")

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Dtypes:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
df.describe()

In [ ]:
for col in ["Fuel_Type", "Selling_type", "Transmission", "Owner"]:
    print(f"\n{col}:", df[col].value_counts().to_dict())

## 2. Data Cleaning

In [ ]:
# 2a. Remove duplicates
df = df.drop_duplicates().reset_index(drop=True)
print(f"After dedup: {df.shape}")

# 2b. Standardise column names
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
print("Columns:", df.columns.tolist())

# 2c. Correct dtypes
for col in ["selling_price", "present_price", "driven_kms", "year", "owner"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 2d. Cap outliers (IQR Winsorisation) for price columns
for col in ["selling_price", "present_price"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = int(((df[col] < q1-1.5*iqr) | (df[col] > q3+1.5*iqr)).sum())
    df[col] = df[col].clip(q1 - 1.5*iqr, q3 + 1.5*iqr)
    print(f"Capped {n_out} outlier(s) in '{col}'")

print(f"\nClean dataset shape: {df.shape}")
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
# Target distribution: raw vs log-transformed
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["selling_price"], bins=30, color="steelblue", edgecolor="white")
axes[0].set(title="Selling Price (Raw)", xlabel="Price (Lakhs INR)", ylabel="Count")
axes[1].hist(np.log1p(df["selling_price"]), bins=30, color="darkorange", edgecolor="white")
axes[1].set(title="log(Selling Price)", xlabel="log(Price)", ylabel="Count")
plt.suptitle("Target Variable Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_target_distribution.png", dpi=150)
plt.show()
print(f"Skewness (raw): {df['selling_price'].skew():.2f}")
print(f"Skewness (log): {np.log1p(df['selling_price']).skew():.2f}")

In [ ]:
# Numerical feature distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["present_price", "driven_kms", "year"]):
    ax.hist(df[col].dropna(), bins=25, color="teal", edgecolor="white")
    ax.set(title=col.replace("_"," ").title(), xlabel=col, ylabel="Count")
plt.suptitle("Numerical Features", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Price vs present price scatter
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["present_price"], df["selling_price"],
           alpha=0.5, color="royalblue", edgecolors="none", s=40)
ax.set(xlabel="Present Price (Lakhs)", ylabel="Selling Price (Lakhs)",
       title="Selling Price vs Present Price")
plt.tight_layout(); plt.show()

In [ ]:
# Categorical comparisons
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ["fuel_type", "transmission", "selling_type"]):
    order = df.groupby(col)["selling_price"].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y="selling_price", order=order, ax=ax, palette="Set2")
    ax.set(title=col.replace("_"," ").title(), xlabel="", ylabel="Selling Price (Lakhs)")
plt.suptitle("Price by Category", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap
corr = df.select_dtypes(include=np.number).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title("Correlation Matrix", fontweight="bold")
plt.tight_layout(); plt.show()

## 4. Feature Engineering

In [ ]:
CURRENT_YEAR = 2024

df["vehicle_age"]      = CURRENT_YEAR - df["year"]
df["depreciation_ratio"] = df["selling_price"] / (df["present_price"] + 1e-9)  # EDA only
df["kms_per_year"]     = df["driven_kms"] / df["vehicle_age"].clip(lower=1)
df["brand_popularity"] = df["car_name"].map(df["car_name"].value_counts()) / len(df)
df["log_selling_price"]= np.log1p(df["selling_price"])   # model target
df["log_driven_kms"]   = np.log1p(df["driven_kms"])

# Encoding
df = pd.get_dummies(df, columns=["fuel_type", "selling_type"], drop_first=True)
df["transmission_encoded"] = (df["transmission"].str.strip().str.lower() == "manual").astype(int)
df = df.drop(columns=["transmission"])

print("Engineered shape:", df.shape)
print("New columns:", [c for c in df.columns if c not in
      ["car_name","year","selling_price","present_price","driven_kms","owner"]])

## 5. Feature Selection & Train/Test Split

In [ ]:
DROP = ["car_name", "year", "selling_price", "depreciation_ratio", "log_selling_price"]
DROP = [c for c in DROP if c in df.columns]

y = df["log_selling_price"]
X = df.drop(columns=DROP)
feature_names = X.columns.tolist()
print("Features:", feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"\nTrain: {X_train.shape[0]} rows  |  Test: {X_test.shape[0]} rows")

## 6. Train Multiple Models

In [ ]:
models = {
    "Linear Regression": Pipeline([("sc", StandardScaler()), ("m", LinearRegression())]),
    "Ridge Regression":  Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0, random_state=RANDOM_STATE))]),
    "Lasso Regression":  Pipeline([("sc", StandardScaler()), ("m", Lasso(alpha=0.01, max_iter=10_000, random_state=RANDOM_STATE))]),
    "Decision Tree":     Pipeline([("sc", StandardScaler()), ("m", DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE))]),
    "Random Forest":     Pipeline([("sc", StandardScaler()), ("m", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE))]),
    "Gradient Boosting": Pipeline([("sc", StandardScaler()), ("m", GradientBoostingRegressor(n_estimators=200, random_state=RANDOM_STATE))]),
}

trained = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    trained[name] = pipe
    print(f"Trained: {name}")

## 7. Model Evaluation

In [ ]:
records = []
for name, pipe in trained.items():
    yp = pipe.predict(X_test)
    records.append({
        "Model": name,
        "R2":    r2_score(y_test, yp),
        "MAE":   mean_absolute_error(y_test, yp),
        "MSE":   mean_squared_error(y_test, yp),
        "RMSE":  float(np.sqrt(mean_squared_error(y_test, yp))),
    })
results = pd.DataFrame(records).sort_values("R2", ascending=False).reset_index(drop=True)
results

In [ ]:
# Model comparison bar chart
fig, ax = plt.subplots(figsize=(9, 4))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(results)))
bars = ax.barh(results["Model"], results["R2"], color=colors[::-1], edgecolor="white")
ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=10)
ax.set(xlabel="R² Score", title="Model Comparison — R² (Test Set)", xlim=(0, 1.05))
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_model_comparison.png", dpi=150)
plt.show()

## 8. Diagnostic Plots — Best Model

In [ ]:
best_name = results.iloc[0]["Model"]
best_pipe = trained[best_name]
yp = best_pipe.predict(X_test)
residuals = np.array(y_test) - yp

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted
lim = [min(y_test.min(), yp.min())-0.1, max(y_test.max(), yp.max())+0.1]
axes[0].scatter(y_test, yp, alpha=0.5, color="steelblue", s=40, edgecolors="none")
axes[0].plot(lim, lim, "r--", linewidth=1.5, label="Perfect")
axes[0].set(xlabel="Actual log(Price)", ylabel="Predicted log(Price)",
            title=f"Actual vs Predicted — {best_name}", xlim=lim, ylim=lim)
axes[0].legend()

# Residual distribution
axes[1].hist(residuals, bins=25, color="darkorange", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set(xlabel="Residual", ylabel="Count", title="Residual Distribution")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "07_diagnostics.png", dpi=150)
plt.show()
print(f"Best model: {best_name}  |  R2={results.iloc[0]['R2']:.4f}")

## 9. Hyperparameter Tuning — Random Forest

In [ ]:
param_dist = {
    "m__n_estimators":      [100, 200, 300, 500],
    "m__max_depth":         [None, 5, 10, 15, 20],
    "m__min_samples_split": [2, 5, 10],
    "m__min_samples_leaf":  [1, 2, 4],
    "m__max_features":      ["sqrt", "log2", None],
}
base_rf = Pipeline([("sc", StandardScaler()),
                    ("m", RandomForestRegressor(random_state=RANDOM_STATE))])
search = RandomizedSearchCV(base_rf, param_dist, n_iter=30, cv=5,
                            scoring="r2", random_state=RANDOM_STATE, verbose=0)
search.fit(X_train, y_train)
tuned_pipe = search.best_estimator_
yp_tuned   = tuned_pipe.predict(X_test)

print("Best params:", search.best_params_)
print(f"CV R2 (best): {search.best_score_:.4f}")
print(f"Test R2     : {r2_score(y_test, yp_tuned):.4f}")
print(f"Test RMSE   : {float(np.sqrt(mean_squared_error(y_test, yp_tuned))):.4f}")

## 10. Feature Importance

In [ ]:
rf_model = trained["Random Forest"].named_steps["m"]
importances = rf_model.feature_importances_
fi = (pd.DataFrame({"Feature": feature_names, "Importance": importances})
      .sort_values("Importance", ascending=True))

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(fi["Feature"], fi["Importance"], color="steelblue", edgecolor="white")
ax.set(xlabel="Importance", title="Feature Importances — Random Forest")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "11_feature_importance.png", dpi=150)
plt.show()
print("\nTop 3 features:")
print(fi.tail(3)[["Feature","Importance"]].to_string(index=False))

## 11. Save Best Model & Verify

In [ ]:
# Decide which model to save: best test R2 wins
test_r2_tuned = r2_score(y_test, yp_tuned)
test_r2_best  = results.iloc[0]["R2"]

if test_r2_tuned > test_r2_best:
    final_pipe = tuned_pipe
    final_name = "Random Forest (Tuned)"
else:
    final_pipe = best_pipe
    final_name = best_name

joblib.dump(final_pipe,    MODELS_DIR / "best_model.pkl")
joblib.dump(tuned_pipe,    MODELS_DIR / "tuned_random_forest.pkl")
joblib.dump(feature_names, MODELS_DIR / "feature_names.pkl")
print(f"Saved '{final_name}' as best_model.pkl")

# Verification round-trip
reloaded   = joblib.load(MODELS_DIR / "best_model.pkl")
log_preds  = reloaded.predict(X_test.iloc[:5])
price_pred = np.expm1(log_preds)
price_act  = np.expm1(y_test.iloc[:5].values)
print("\nSample verification (Lakhs INR):")
for a, p in zip(price_act, price_pred):
    print(f"  Actual: {a:6.2f}  |  Predicted: {p:6.2f}")

## 12. Business Insights & Real-World Applications

| Factor | Finding |
|---|---|
| **Present Price** | Strongest predictor — resale tracks new price closely |
| **Vehicle Age** | Each additional year lowers resale value significantly |
| **Driven KMs** | Higher mileage consistently reduces price |
| **Fuel Type** | Diesel commands a premium in the Indian resale market |
| **Transmission** | Automatic cars sell for more on average |
| **Seller Type** | Dealer listings trend higher than individual sellers |

### Applications
- **Used car platforms** — automated fair-price recommendations
- **Insurance** — vehicle valuation for comprehensive cover
- **Auto financing** — LTV ratio for loan approval
- **Dealerships** — dynamic pricing and inventory management
